<a href="https://colab.research.google.com/github/anushah-200/SATARK_AI/blob/main/notebooks/Person2_AnomalyDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive

drive.mount('/content/drive')

print("Google Drive mounted successfully!")

Mounted at /content/drive
Google Drive mounted successfully!


In [3]:
import os

print(os.listdir('/content/drive/MyDrive'))

['obstacle1.png', 'ground2.png', 'style.css', 'jump.mp3', 'trex_collided.png', 'index.html', 'README.md', 'obstacle6.png', 'gameOver.png', 'obstacle3.png', 'p5.play.js', 'restart.png', 'checkPoint.mp3', 'obstacle5.png', 'sketch.js', 'p5.js', 'trex1.png', 'obstacle4.png', 'trex4.png', 'die.mp3', 'obstacle2.png', 'cloud.png', 'trex3.png', 'p5.sound.min.js', 'p5.dom.min.js', 'jb and the other jb dec(2023).gsheet', 'Annual budget.gsheet', 'jee main.gsheet', 'eb382ade-dd65-443b-aefe-3e66d157ec51.jpeg', 'allen Chemistry Module (1) (1).pdf', 'october to january.gsheet', 'advanced.gsheet', 'IMG_9329 (1).JPG', 'IMG_9330.JPG', 'IMG_9331.JPG', 'IMG_9332.JPG', 'IMG_9333.JPG', 'IMG_9461 (1).png', 'IMG_9461.png', '02201012025.jpg', 'select projects TechNeeds form', 'stamp.HEIC', 'pRUwwmEg44MxYM8F9', 'IMG_9533.JPG', 'Untitled document (14).gdoc', '02201012025 (1).gdoc', 'experiment 3 text format.gdoc', 'Process Has Intrinsic Value by Anushah Gupta.gdoc', 'Image 18-10-25 at 11.19 PM.jpeg', 'image0 (1)

In [4]:
PROJECT_PATH = "/content/drive/MyDrive/SATARK_AI"

print("Project folder exists:", os.path.exists(PROJECT_PATH))

Project folder exists: True


In [5]:
for root, dirs, files in os.walk(PROJECT_PATH):
    level = root.replace(PROJECT_PATH, '').count(os.sep)
    indent = '    ' * level
    print(indent + os.path.basename(root) + '/')

    subindent = '    ' * (level + 1)
    for file in files:
        print(subindent + file)

SATARK_AI/
    data/
        raw/
            meteostat_data.csv
        processed/
            processed_meteostat_data.csv
        synthetic/
            synthetic_fault_data.csv
            processed_meteostat_data (1).csv
            p2_day2_handoff.csv
            demo_faults.csv
            day2_demo_faults.csv
    outputs/
        figures/
        results/
            anomaly_detection_results.csv
    src/
        anomaly_detector.py
        evaluate_model.py
        spatial_analysis.py
        diagnosis.py
        person3_pipeline.py
        sensor_health.py
        fault_injection.py
        data_preprocessing.py
    notebooks/
        01_EDA.ipynb
        02_Preprocessing.ipynb
        03_Fault_Injection.ipynb
        P2.ipynb
    models/
        isolation_forest.pkl
        scaler.pkl


In [93]:
import os

PROJECT_PATH = "/content/drive/MyDrive/SATARK_AI"

DATA_PATH = "/content/drive/MyDrive/SATARK_AI/data/synthetic/p2_day2_handoff.csv"

RESULTS_PATH = "/content/drive/MyDrive/SATARK_AI/outputs/results/person2_day2_results.csv"

print("Project path:", PROJECT_PATH)
print("Data path:", DATA_PATH)
print("Results path:", RESULTS_PATH)

Project path: /content/drive/MyDrive/SATARK_AI
Data path: /content/drive/MyDrive/SATARK_AI/data/synthetic/p2_day2_handoff.csv
Results path: /content/drive/MyDrive/SATARK_AI/outputs/results/person2_day2_results.csv


In [9]:
print("Dataset exists:", os.path.exists(DATA_PATH))

Dataset exists: True


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (43100, 36)


In [11]:
print(df.head())

             timestamp  station_id station_name  temperature  \
0  2025-01-01 00:00:00       42131       Hissar          5.4   
1  2025-01-01 01:00:00       42131       Hissar          6.1   
2  2025-01-01 02:00:00       42131       Hissar          6.1   
3  2025-01-01 03:00:00       42131       Hissar          8.0   
4  2025-01-01 04:00:00       42131       Hissar          7.6   

   original_temperature  humidity  original_humidity  pressure  \
0                   5.4      97.0               97.0    1020.7   
1                   6.1      98.0               98.0    1019.5   
2                   6.1      94.0               94.0    1019.8   
3                   8.0      97.0               97.0    1021.5   
4                   7.6      96.0               96.0    1020.8   

   original_pressure  wind_speed  ...  humidity_rolling_std_24h  \
0             1020.7         0.0  ...                       NaN   
1             1019.5         5.0  ...                       NaN   
2             101

In [12]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['timestamp', 'station_id', 'station_name', 'temperature', 'original_temperature', 'humidity', 'original_humidity', 'pressure', 'original_pressure', 'wind_speed', 'wind_direction', 'hour', 'day', 'month', 'day_of_week', 'day_of_year', 'time_period', 'temperature_rolling_mean_24h', 'temperature_rolling_std_24h', 'temperature_deviation_24h', 'temperature_zscore_24h', 'pressure_rolling_mean_24h', 'pressure_rolling_std_24h', 'pressure_deviation_24h', 'pressure_zscore_24h', 'humidity_rolling_mean_24h', 'humidity_rolling_std_24h', 'humidity_deviation_24h', 'humidity_zscore_24h', 'fault_type', 'fault_severity', 'is_anomaly', 'temperature_rule_flag', 'humidity_rule_flag', 'pressure_rule_flag', 'qc_flag']


In [13]:
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Missing values:
temperature                       1
wind_direction                  150
temperature_rolling_mean_24h     15
temperature_rolling_std_24h      15
temperature_deviation_24h        15
temperature_zscore_24h           15
pressure_rolling_mean_24h        15
pressure_rolling_std_24h         15
pressure_deviation_24h           15
pressure_zscore_24h              15
humidity_rolling_mean_24h        15
humidity_rolling_std_24h         15
humidity_deviation_24h           15
humidity_zscore_24h              16
dtype: int64


In [14]:
anomalies = df[df["is_anomaly"] == 1].copy()

print("Number of injected anomalies:", len(anomalies))

print("\nFault distribution:")
print(anomalies["fault_type"].value_counts())

print("\nInjected anomalies:")
print(
    anomalies[
        [
            "timestamp",
            "station_id",
            "station_name",
            "temperature",
            "humidity",
            "pressure",
            "fault_type",
            "fault_severity",
            "is_anomaly"
        ]
    ].to_string(index=False)
)

Number of injected anomalies: 20

Fault distribution:
fault_type
temperature_frozen     6
temperature_noise      6
temperature_drift      6
temperature_spike      1
temperature_missing    1
Name: count, dtype: int64

Injected anomalies:
          timestamp  station_id station_name  temperature  humidity  pressure          fault_type fault_severity  is_anomaly
2025-01-05 04:00:00       42182   Safdarjung    55.000000      92.0    1017.7   temperature_spike         severe           1
2025-01-09 08:00:00       42182   Safdarjung          NaN      58.0    1018.0 temperature_missing       moderate           1
2025-01-13 12:00:00       42182   Safdarjung    16.200000      81.0    1016.6  temperature_frozen         severe           1
2025-01-13 13:00:00       42182   Safdarjung    16.200000      81.0    1017.2  temperature_frozen         severe           1
2025-01-13 14:00:00       42182   Safdarjung    16.200000      86.0    1017.7  temperature_frozen         severe           1
2025-01-13 15

In [15]:
print("QC flag distribution:")
print(df["qc_flag"].value_counts())

print("\nInjected anomalies and their QC flags:")
print(
    anomalies[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "temperature_rule_flag",
            "qc_flag"
        ]
    ].to_string(index=False)
)

QC flag distribution:
qc_flag
0    43099
1        1
Name: count, dtype: int64

Injected anomalies and their QC flags:
          timestamp          fault_type  temperature  temperature_rule_flag  qc_flag
2025-01-05 04:00:00   temperature_spike    55.000000                      0        0
2025-01-09 08:00:00 temperature_missing          NaN                      1        1
2025-01-13 12:00:00  temperature_frozen    16.200000                      0        0
2025-01-13 13:00:00  temperature_frozen    16.200000                      0        0
2025-01-13 14:00:00  temperature_frozen    16.200000                      0        0
2025-01-13 15:00:00  temperature_frozen    16.200000                      0        0
2025-01-13 16:00:00  temperature_frozen    16.200000                      0        0
2025-01-13 17:00:00  temperature_frozen    16.200000                      0        0
2025-01-17 16:00:00   temperature_drift    11.100000                      0        0
2025-01-17 17:00:00   temperatur

In [17]:
df["rule_score"] = 0

# Temperature physical range
temperature_rule = (
    (df["temperature"] < -50) |
    (df["temperature"] > 60) |
    (df["temperature"].isna())
)

# Humidity physical range
humidity_rule = (
    (df["humidity"] < 0) |
    (df["humidity"] > 100) |
    (df["humidity"].isna())
)

# Pressure physical range
pressure_rule = (
    (df["pressure"] < 870) |
    (df["pressure"] > 1085) |
    (df["pressure"].isna())
)

# If any physical rule is violated
df["rule_score"] = (
    temperature_rule |
    humidity_rule |
    pressure_rule
).astype(int)

print("Rule-based detections:")
print(df["rule_score"].value_counts())

Rule-based detections:
rule_score
0    43099
1        1
Name: count, dtype: int64


In [18]:
rule_anomalies = df[df["rule_score"] == 1]

print("Total rule-based detections:", len(rule_anomalies))

print(
    "\nInjected anomalies detected by rules:",
    (
        df[
            (df["is_anomaly"] == 1) &
            (df["rule_score"] == 1)
        ]
        .shape[0]
    )
)

print("\nFaults detected:")
print(
    df[
        (df["is_anomaly"] == 1) &
        (df["rule_score"] == 1)
    ][
        ["timestamp", "fault_type", "temperature", "rule_score"]
    ]
)

Total rule-based detections: 1

Injected anomalies detected by rules: 1

Faults detected:
                 timestamp           fault_type  temperature  rule_score
34539  2025-01-09 08:00:00  temperature_missing          NaN           1


In [19]:
# Inspect statistical features for the injected anomalies

print(
    anomalies[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "temperature_deviation_24h",
            "temperature_zscore_24h"
        ]
    ].to_string(index=False)
)

          timestamp          fault_type  temperature  temperature_deviation_24h  temperature_zscore_24h
2025-01-05 04:00:00   temperature_spike    55.000000                  -1.975000               -0.615017
2025-01-09 08:00:00 temperature_missing          NaN                   6.087500                1.244358
2025-01-13 12:00:00  temperature_frozen    16.200000                   3.162500                1.321824
2025-01-13 13:00:00  temperature_frozen    16.200000                   2.237500                0.923289
2025-01-13 14:00:00  temperature_frozen    16.200000                   1.204167                0.491618
2025-01-13 15:00:00  temperature_frozen    16.200000                   0.875000                0.355719
2025-01-13 16:00:00  temperature_frozen    16.200000                   0.233333                0.094618
2025-01-13 17:00:00  temperature_frozen    16.200000                  -0.400000               -0.162372
2025-01-17 16:00:00   temperature_drift    11.100000            

In [20]:
print("\nAbsolute temperature z-scores for injected anomalies:")

print(
    anomalies[
        ["fault_type", "temperature_zscore_24h"]
    ]
    .assign(
        abs_zscore=lambda x: x["temperature_zscore_24h"].abs()
    )
    .to_string(index=False)
)


Absolute temperature z-scores for injected anomalies:
         fault_type  temperature_zscore_24h  abs_zscore
  temperature_spike               -0.615017    0.615017
temperature_missing                1.244358    1.244358
 temperature_frozen                1.321824    1.321824
 temperature_frozen                0.923289    0.923289
 temperature_frozen                0.491618    0.491618
 temperature_frozen                0.355719    0.355719
 temperature_frozen                0.094618    0.094618
 temperature_frozen               -0.162372    0.162372
  temperature_drift               -0.561838    0.561838
  temperature_drift               -0.619879    0.619879
  temperature_drift               -0.286821    0.286821
  temperature_drift               -0.877529    0.877529
  temperature_drift               -0.996880    0.996880
  temperature_drift               -0.344534    0.344534
  temperature_noise               -0.796862    0.796862
  temperature_noise               -1.426636    1.

In [21]:
# Inspect z-score distribution for normal observations

normal = df[df["is_anomaly"] == 0]

normal_abs_z = normal["temperature_zscore_24h"].abs()

print("Normal temperature |z-score| distribution:")
print(
    normal_abs_z.describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

Normal temperature |z-score| distribution:
count    43065.000000
mean         0.891006
std          0.544467
min          0.000000
50%          0.893178
90%          1.531312
95%          1.697709
99%          2.210402
99.5%        2.435899
99.9%        3.156302
max         32.331615
Name: temperature_zscore_24h, dtype: float64


In [22]:
# Compare normal and injected anomaly z-score ranges

print("Normal |z-score|:")
print("95th percentile :", normal_abs_z.quantile(0.95))
print("99th percentile :", normal_abs_z.quantile(0.99))
print("99.5th percentile:", normal_abs_z.quantile(0.995))
print("99.9th percentile:", normal_abs_z.quantile(0.999))

print("\nInjected anomaly |z-score|:")
print(
    anomalies["temperature_zscore_24h"]
    .abs()
    .describe()
)

Normal |z-score|:
95th percentile : 1.6977086700434745
99th percentile : 2.210401975189886
99.5th percentile: 2.435899476895202
99.9th percentile: 3.1563024951318503

Injected anomaly |z-score|:
count    20.000000
mean      0.814395
std       0.441011
min       0.094618
25%       0.457643
50%       0.837196
75%       1.253897
max       1.426636
Name: temperature_zscore_24h, dtype: float64


In [23]:
# Inspect temperature deviation for normal vs injected anomalies

normal_deviation = normal["temperature_deviation_24h"].abs()
anomaly_deviation = anomalies["temperature_deviation_24h"].abs()

print("NORMAL |temperature deviation|")
print(
    normal_deviation.describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

print("\nINJECTED ANOMALIES |temperature deviation|")
print(anomaly_deviation.describe())

NORMAL |temperature deviation|
count    43065.000000
mean         3.269733
std          2.178906
min          0.000000
50%          2.941667
90%          6.375000
95%          7.308333
99%          8.800000
99.5%        9.362500
99.9%       10.515600
max         13.208333
Name: temperature_deviation_24h, dtype: float64

INJECTED ANOMALIES |temperature deviation|
count    20.000000
mean      2.838125
std       1.986738
min       0.233333
25%       1.150000
50%       2.350000
75%       5.130208
max       6.087500
Name: temperature_deviation_24h, dtype: float64


In [24]:
# Compare deviation by fault type

print(
    anomalies[
        ["fault_type", "temperature_deviation_24h"]
    ]
    .assign(
        abs_deviation=lambda x: x["temperature_deviation_24h"].abs()
    )
    .groupby("fault_type")["abs_deviation"]
    .agg(["count", "min", "mean", "max"])
)

                     count       min      mean       max
fault_type                                              
temperature_drift        6  0.812500  1.734722  2.825000
temperature_frozen       6  0.233333  1.352083  3.162500
temperature_missing      1  6.087500  6.087500  6.087500
temperature_noise        6  3.108333  5.029861  5.733333
temperature_spike        1  1.975000  1.975000  1.975000


In [25]:
# Inspect rolling standard deviation for normal vs injected anomalies

normal_std = normal["temperature_rolling_std_24h"]
anomaly_std = anomalies["temperature_rolling_std_24h"]

print("NORMAL temperature rolling std:")
print(
    normal_std.describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

print("\nINJECTED ANOMALIES temperature rolling std:")
print(anomaly_std.describe())

print("\nInjected anomalies by fault type:")
print(
    anomalies[
        ["fault_type", "temperature_rolling_std_24h"]
    ]
    .groupby("fault_type")["temperature_rolling_std_24h"]
    .agg(["count", "min", "mean", "max"])
)

NORMAL temperature rolling std:
count    43065.000000
mean         3.733596
std          1.299871
min          0.057735
50%          3.857514
90%          5.355286
95%          5.808290
99%          6.595222
99.5%        6.756691
99.9%        6.989604
max          8.056623
Name: temperature_rolling_std_24h, dtype: float64

INJECTED ANOMALIES temperature rolling std:
count    20.000000
mean      3.208600
std       0.790287
min       2.392527
25%       2.465413
50%       2.833308
75%       3.946639
max       4.892080
Name: temperature_rolling_std_24h, dtype: float64

Injected anomalies by fault type:
                     count       min      mean       max
fault_type                                              
temperature_drift        6  2.803298  2.826448  2.866192
temperature_frozen       6  2.392527  2.442444  2.466059
temperature_missing      1  4.892080  4.892080  4.892080
temperature_noise        6  3.900715  4.075878  4.271875
temperature_spike        1  3.211291  3.211291  3.21

In [26]:
# Check false-positive rates for possible statistical thresholds

print("Temperature deviation thresholds:")
for threshold in [5, 6, 7, 8, 9, 10]:
    flagged = (normal["temperature_deviation_24h"].abs() > threshold).sum()
    rate = flagged / len(normal) * 100
    print(f"> {threshold:2} : {flagged:5} normal rows ({rate:.3f}%)")

print("\nAbsolute z-score thresholds:")
for threshold in [1.5, 2, 2.5, 3]:
    flagged = (normal["temperature_zscore_24h"].abs() > threshold).sum()
    rate = flagged / len(normal) * 100
    print(f"> {threshold:3} : {flagged:5} normal rows ({rate:.3f}%)")

Temperature deviation thresholds:
>  5 :  9594 normal rows (22.270%)
>  6 :  5476 normal rows (12.711%)
>  7 :  2724 normal rows (6.323%)
>  8 :  1084 normal rows (2.516%)
>  9 :   340 normal rows (0.789%)
> 10 :    92 normal rows (0.214%)

Absolute z-score thresholds:
> 1.5 :  4890 normal rows (11.351%)
>   2 :   793 normal rows (1.841%)
> 2.5 :   188 normal rows (0.436%)
>   3 :    61 normal rows (0.142%)


In [27]:
# Statistical anomaly detection
# Conservative thresholds chosen from the normal-data distribution

df["statistical_score"] = (
    (
        df["temperature_zscore_24h"].abs() > 3
    )
    |
    (
        df["temperature_deviation_24h"].abs() > 10
    )
).fillna(False).astype(int)

print("Statistical detections:")
print(df["statistical_score"].value_counts())

print("\nInjected anomalies detected statistically:")

stat_detected = df[
    (df["is_anomaly"] == 1) &
    (df["statistical_score"] == 1)
]

print("Detected:", len(stat_detected), "/ 20")

print(
    stat_detected[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "temperature_deviation_24h",
            "temperature_zscore_24h",
            "statistical_score"
        ]
    ].to_string(index=False)
)

Statistical detections:
statistical_score
0    42952
1      148
Name: count, dtype: int64

Injected anomalies detected statistically:
Detected: 0 / 20
Empty DataFrame
Columns: [timestamp, fault_type, temperature, temperature_deviation_24h, temperature_zscore_24h, statistical_score]
Index: []


In [28]:
# Statistical anomaly detection
# Conservative thresholds based on the normal-data distribution

df["statistical_score"] = (
    (
        df["temperature_zscore_24h"].abs() > 3
    )
    |
    (
        df["temperature_deviation_24h"].abs() > 10
    )
).fillna(False).astype(int)

print("Statistical detections:")
print(df["statistical_score"].value_counts())

stat_detected = df[
    (df["is_anomaly"] == 1) &
    (df["statistical_score"] == 1)
]

print("\nInjected anomalies detected statistically:")
print("Detected:", len(stat_detected), "/ 20")

if len(stat_detected) > 0:
    print(
        stat_detected[
            [
                "timestamp",
                "fault_type",
                "temperature",
                "temperature_deviation_24h",
                "temperature_zscore_24h",
                "statistical_score"
            ]
        ].to_string(index=False)
    )

Statistical detections:
statistical_score
0    42952
1      148
Name: count, dtype: int64

Injected anomalies detected statistically:
Detected: 0 / 20


In [29]:
# Prepare features for Isolation Forest

if_features = [
    "temperature",
    "humidity",
    "pressure",
    "wind_speed",
    "temperature_deviation_24h",
    "temperature_zscore_24h",
    "pressure_deviation_24h",
    "pressure_zscore_24h",
    "humidity_deviation_24h",
    "humidity_zscore_24h"
]

X_if = df[if_features].copy()

# Replace infinite values with NaN
X_if = X_if.replace([np.inf, -np.inf], np.nan)

# Fill missing values using training-feature medians
X_if = X_if.fillna(X_if.median())

print("Isolation Forest feature matrix created.")
print("Shape:", X_if.shape)

print("\nFeatures:")
print(if_features)

print("\nRemaining missing values:")
print(X_if.isna().sum().sum())

Isolation Forest feature matrix created.
Shape: (43100, 10)

Features:
['temperature', 'humidity', 'pressure', 'wind_speed', 'temperature_deviation_24h', 'temperature_zscore_24h', 'pressure_deviation_24h', 'pressure_zscore_24h', 'humidity_deviation_24h', 'humidity_zscore_24h']

Remaining missing values:
0


In [30]:
# Train Isolation Forest

isolation_forest = IsolationForest(
    n_estimators=200,
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(X_if)

print("Isolation Forest trained successfully!")

Isolation Forest trained successfully!


In [31]:
# Generate Isolation Forest predictions

if_predictions = isolation_forest.predict(X_if)

df["isolation_score"] = (if_predictions == -1).astype(int)

print("Isolation Forest detections:")
print(df["isolation_score"].value_counts())

# Check detection of injected anomalies
if_detected = df[
    (df["is_anomaly"] == 1) &
    (df["isolation_score"] == 1)
]

print("\nInjected anomalies detected:")
print("Detected:", len(if_detected), "/ 20")

print("\nDetection by fault type:")
print(
    if_detected["fault_type"].value_counts()
)

print("\nDetected anomaly rows:")
print(
    if_detected[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "isolation_score"
        ]
    ].to_string(index=False)
)

Isolation Forest detections:
isolation_score
0    35888
1     7212
Name: count, dtype: int64

Injected anomalies detected:
Detected: 2 / 20

Detection by fault type:
fault_type
temperature_frozen    1
temperature_noise     1
Name: count, dtype: int64

Detected anomaly rows:
          timestamp         fault_type  temperature  isolation_score
2025-01-13 12:00:00 temperature_frozen     16.20000                1
2025-01-22 00:00:00  temperature_noise      6.49793                1


In [32]:
# Retrain Isolation Forest with lower contamination

isolation_forest_low = IsolationForest(
    n_estimators=200,
    contamination=0.005,   # 0.5%
    random_state=42,
    n_jobs=-1
)

isolation_forest_low.fit(X_if)

# Generate predictions
if_predictions_low = isolation_forest_low.predict(X_if)

df["isolation_score_low"] = (
    if_predictions_low == -1
).astype(int)

print("Isolation Forest detections:")
print(df["isolation_score_low"].value_counts())

# Evaluate against injected anomalies
if_detected_low = df[
    (df["is_anomaly"] == 1) &
    (df["isolation_score_low"] == 1)
]

print("\nInjected anomalies detected:")
print("Detected:", len(if_detected_low), "/ 20")

print("\nDetection by fault type:")
print(
    if_detected_low["fault_type"].value_counts()
)

print("\nDetected anomaly rows:")
print(
    if_detected_low[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "isolation_score_low"
        ]
    ].to_string(index=False)
)

Isolation Forest detections:
isolation_score_low
0    42884
1      216
Name: count, dtype: int64

Injected anomalies detected:
Detected: 0 / 20

Detection by fault type:
Series([], Name: count, dtype: int64)

Detected anomaly rows:
Empty DataFrame
Columns: [timestamp, fault_type, temperature, isolation_score_low]
Index: []


In [33]:
# Inspect Isolation Forest anomaly scores

df["if_raw_score"] = isolation_forest.decision_function(X_if)

# Lower score = more anomalous
print("Isolation Forest score statistics:")
print(df["if_raw_score"].describe())

print("\nInjected anomaly scores:")
print(
    df[df["is_anomaly"] == 1][
        [
            "timestamp",
            "fault_type",
            "temperature",
            "if_raw_score"
        ]
    ]
    .sort_values("if_raw_score")
    .to_string(index=False)
)

Isolation Forest score statistics:
count    43100.000000
mean         0.037277
std          0.038699
min         -0.186237
25%          0.014250
50%          0.043367
75%          0.066542
max          0.109289
Name: if_raw_score, dtype: float64

Injected anomaly scores:
          timestamp          fault_type  temperature  if_raw_score
2025-01-22 00:00:00   temperature_noise     6.497930     -0.038553
2025-01-13 12:00:00  temperature_frozen    16.200000     -0.004527
2025-01-21 21:00:00   temperature_noise     8.720032      0.002781
2025-01-09 08:00:00 temperature_missing          NaN      0.005339
2025-01-05 04:00:00   temperature_spike    55.000000      0.013304
2025-01-13 13:00:00  temperature_frozen    16.200000      0.020072
2025-01-21 22:00:00   temperature_noise    12.400902      0.025915
2025-01-21 23:00:00   temperature_noise    12.781129      0.038163
2025-01-17 21:00:00   temperature_drift    16.600000      0.039242
2025-01-22 01:00:00   temperature_noise     8.395641      

In [34]:
# Compare injected anomalies with the most anomalous normal observations

print("10 most anomalous NORMAL observations:")

print(
    df[df["is_anomaly"] == 0][
        [
            "timestamp",
            "temperature",
            "humidity",
            "pressure",
            "if_raw_score"
        ]
    ]
    .sort_values("if_raw_score")
    .head(10)
    .to_string(index=False)
)

10 most anomalous NORMAL observations:
          timestamp  temperature  humidity  pressure  if_raw_score
2025-05-02 01:00:00         20.0     100.0    1016.0     -0.186237
2025-05-24 21:00:00         22.0      94.0    1005.0     -0.185128
2025-05-02 00:00:00         19.0     100.0    1013.0     -0.184507
2025-05-21 15:00:00         24.0      89.0    1005.0     -0.184481
2025-06-01 11:00:00         24.0      89.0    1002.0     -0.160854
2025-05-02 02:00:00         20.0     100.0    1014.0     -0.160076
2025-12-06 08:00:00         23.0      27.0    1018.0     -0.136445
2025-01-08 08:00:00         19.0      43.0    1015.0     -0.134989
2025-05-24 20:00:00         25.0      74.0    1003.0     -0.132618
2025-07-15 10:00:00         32.7      61.0     994.2     -0.132205


In [35]:
#Calculate temporal temperature changes

# Make sure timestamps are properly formatted
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort by station and time
df = df.sort_values(
    ["station_id", "timestamp"]
).reset_index(drop=True)

# Calculate change from the previous observation at the same station
df["temperature_change"] = (
    df.groupby("station_id")["temperature"]
    .diff()
)

# Absolute change
df["absolute_temperature_change"] = (
    df["temperature_change"].abs()
)

print("Temporal features created.")

print(
    df[
        [
            "station_id",
            "timestamp",
            "temperature",
            "temperature_change",
            "absolute_temperature_change"
        ]
    ].head(10).to_string(index=False)
)

Temporal features created.
 station_id           timestamp  temperature  temperature_change  absolute_temperature_change
      42131 2025-01-01 00:00:00          5.4                 NaN                          NaN
      42131 2025-01-01 01:00:00          6.1                 0.7                          0.7
      42131 2025-01-01 02:00:00          6.1                 0.0                          0.0
      42131 2025-01-01 03:00:00          8.0                 1.9                          1.9
      42131 2025-01-01 04:00:00          7.6                -0.4                          0.4
      42131 2025-01-01 05:00:00          9.8                 2.2                          2.2
      42131 2025-01-01 06:00:00         10.0                 0.2                          0.2
      42131 2025-01-01 07:00:00         14.9                 4.9                          4.9
      42131 2025-01-01 08:00:00         16.3                 1.4                          1.4
      42131 2025-01-01 09:00:00  

In [36]:
# Inspect temporal changes for injected anomalies

temporal_anomalies = df[df["is_anomaly"] == 1].copy()

print(
    temporal_anomalies[
        [
            "timestamp",
            "fault_type",
            "temperature",
            "temperature_change",
            "absolute_temperature_change"
        ]
    ].to_string(index=False)
)

print("\nAbsolute temperature change by fault type:")

print(
    temporal_anomalies
    .assign(
        abs_change=lambda x: x["temperature_change"].abs()
    )
    .groupby("fault_type")["abs_change"]
    .agg(["count", "min", "mean", "max"])
)

          timestamp          fault_type  temperature  temperature_change  absolute_temperature_change
2025-01-05 04:00:00   temperature_spike    55.000000           44.800000                    44.800000
2025-01-09 08:00:00 temperature_missing          NaN                 NaN                          NaN
2025-01-13 12:00:00  temperature_frozen    16.200000           -0.700000                     0.700000
2025-01-13 13:00:00  temperature_frozen    16.200000            0.000000                     0.000000
2025-01-13 14:00:00  temperature_frozen    16.200000            0.000000                     0.000000
2025-01-13 15:00:00  temperature_frozen    16.200000            0.000000                     0.000000
2025-01-13 16:00:00  temperature_frozen    16.200000            0.000000                     0.000000
2025-01-13 17:00:00  temperature_frozen    16.200000            0.000000                     0.000000
2025-01-17 16:00:00   temperature_drift    11.100000           -0.300000          

In [38]:
# Refresh normal and anomaly subsets after creating temporal features

normal = df[df["is_anomaly"] == 0].copy()
anomalies = df[df["is_anomaly"] == 1].copy()

print("Normal rows:", len(normal))
print("Anomaly rows:", len(anomalies))

print("\nTemporal feature available in normal:")
print("temperature_change" in normal.columns)

print("Temporal feature available in anomalies:")
print("temperature_change" in anomalies.columns)

Normal rows: 43080
Anomaly rows: 20

Temporal feature available in normal:
True
Temporal feature available in anomalies:
True


In [39]:
# Inspect normal hourly temperature changes

normal_change = (
    normal["temperature_change"]
    .abs()
    .dropna()
)

print("Normal absolute hourly temperature change:")
print(
    normal_change.describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

print("\nMaximum normal hourly change:")
print(normal_change.max())

Normal absolute hourly temperature change:
count    43074.000000
mean         0.993590
std          1.058696
min          0.000000
50%          0.700000
90%          2.300000
95%          3.000000
99%          4.700000
99.5%        5.700000
99.9%        9.000000
max         42.300000
Name: temperature_change, dtype: float64

Maximum normal hourly change:
42.3


In [40]:
# Check normal observations exceeding candidate change thresholds

print("Normal observations exceeding hourly-change thresholds:")

for threshold in [3, 4, 5, 6, 8, 10]:
    flagged = (normal_change > threshold).sum()
    rate = flagged / len(normal_change) * 100

    print(
        f"> {threshold:2}°C : "
        f"{flagged:5} rows ({rate:.3f}%)"
    )

Normal observations exceeding hourly-change thresholds:
>  3°C :  1808 rows (4.197%)
>  4°C :   662 rows (1.537%)
>  5°C :   324 rows (0.752%)
>  6°C :   176 rows (0.409%)
>  8°C :    69 rows (0.160%)
> 10°C :    29 rows (0.067%)


In [42]:
# Refresh normal and anomaly subsets after adding temporal features

normal = df[df["is_anomaly"] == 0].copy()
anomalies = df[df["is_anomaly"] == 1].copy()

print("Normal rows:", len(normal))
print("Anomaly rows:", len(anomalies))

print("\nColumn available:")
print("same_as_previous" in normal.columns)

Normal rows: 43080
Anomaly rows: 20

Column available:
True


In [43]:
# Check consecutive repeated temperature values

normal_repeat = normal["same_as_previous"].value_counts()

print("Normal consecutive temperature values:")
print(normal_repeat)

print("\nNormal repeated-value percentage:")
print(normal["same_as_previous"].mean() * 100, "%")

Normal consecutive temperature values:
same_as_previous
False    39001
True      4079
Name: count, dtype: int64

Normal repeated-value percentage:
9.468430826369545 %


In [44]:
# Maximum consecutive identical temperature runs in normal data

def max_consecutive_repeats(series):
    groups = (series != series.shift()).cumsum()
    return series.groupby(groups).size().max()

normal_repeat_lengths = (
    normal.groupby("station_id")["temperature"]
    .apply(max_consecutive_repeats)
)

print("Maximum consecutive identical-temperature run per station:")
print(normal_repeat_lengths)

print("\nOverall maximum:")
print(normal_repeat_lengths.max())

Maximum consecutive identical-temperature run per station:
station_id
42131     4
42139     5
42176     4
42181    13
42182     4
Name: temperature, dtype: int64

Overall maximum:
13


In [46]:
# Calculate consecutive identical-temperature run length

df["repeat_group"] = (
    df.groupby("station_id")["temperature"]
      .transform(lambda x: (x != x.shift()).cumsum())
)

df["repeat_length"] = (
    df.groupby(["station_id", "repeat_group"])["temperature"]
      .transform("size")
)

# Directly select normal rows after creating the new feature
normal_repeat_lengths = df.loc[
    df["is_anomaly"] == 0,
    "repeat_length"
]

print("Normal repeat-length distribution:")
print(normal_repeat_lengths.value_counts().sort_index())

Normal repeat-length distribution:
repeat_length
1     36311
2      3914
3      1245
4       680
5       380
6       156
7       140
8        64
9        54
10       20
11       55
12       48
13       13
Name: count, dtype: int64


In [47]:
# Check long repeated-temperature runs in normal data

for threshold in [4, 5, 6, 7, 8, 10]:
    count = (normal_repeat_lengths >= threshold).sum()
    rate = count / len(normal_repeat_lengths) * 100

    print(
        f"Run length >= {threshold}: "
        f"{count} rows ({rate:.3f}%)"
    )

Run length >= 4: 1610 rows (3.737%)
Run length >= 5: 930 rows (2.159%)
Run length >= 6: 550 rows (1.277%)
Run length >= 7: 394 rows (0.915%)
Run length >= 8: 254 rows (0.590%)
Run length >= 10: 136 rows (0.316%)


In [48]:
# Temperature change over 6 hours

df["temperature_change_6h"] = (
    df.groupby("station_id")["temperature"].diff(6)
)

normal_6h_change = df.loc[
    df["is_anomaly"] == 0,
    "temperature_change_6h"
].abs().dropna()

print("Normal absolute 6-hour temperature change:")
print(
    normal_6h_change.describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

Normal absolute 6-hour temperature change:
count    43049.000000
mean         4.601027
std          3.290035
min          0.000000
50%          4.000000
90%          9.300000
95%         11.100000
99%         14.000000
99.5%       15.000000
99.9%       16.895200
max         37.400000
Name: temperature_change_6h, dtype: float64


In [49]:
# False-positive rates for different 6-hour change thresholds

for threshold in [3, 4, 5, 6, 7, 8, 10]:
    count = (normal_6h_change > threshold).sum()
    rate = count / len(normal_6h_change) * 100

    print(
        f"> {threshold:2}°C : "
        f"{count:5} rows ({rate:.3f}%)"
    )

>  3°C : 26035 rows (60.478%)
>  4°C : 20141 rows (46.786%)
>  5°C : 15553 rows (36.129%)
>  6°C : 11962 rows (27.787%)
>  7°C :  9073 rows (21.076%)
>  8°C :  6606 rows (15.345%)
> 10°C :  3242 rows (7.531%)


In [50]:
# 6-hour change for injected anomalies

print(
    df.loc[
        df["is_anomaly"] == 1,
        [
            "timestamp",
            "fault_type",
            "temperature",
            "temperature_change_6h"
        ]
    ].to_string(index=False)
)

          timestamp          fault_type  temperature  temperature_change_6h
2025-01-05 04:00:00   temperature_spike    55.000000              44.600000
2025-01-09 08:00:00 temperature_missing          NaN                    NaN
2025-01-13 12:00:00  temperature_frozen    16.200000               2.200000
2025-01-13 13:00:00  temperature_frozen    16.200000               1.200000
2025-01-13 14:00:00  temperature_frozen    16.200000               0.000000
2025-01-13 15:00:00  temperature_frozen    16.200000              -1.800000
2025-01-13 16:00:00  temperature_frozen    16.200000              -1.200000
2025-01-13 17:00:00  temperature_frozen    16.200000              -0.700000
2025-01-17 16:00:00   temperature_drift    11.100000              -5.800000
2025-01-17 17:00:00   temperature_drift    11.900000              -4.800000
2025-01-17 18:00:00   temperature_drift    13.800000              -1.400000
2025-01-17 19:00:00   temperature_drift    13.200000              -1.900000
2025-01-17 2

In [51]:
# Calculate 6-hour rolling temperature slope
# Positive = increasing temperature
# Negative = decreasing temperature

def rolling_slope(values):
    values = np.array(values, dtype=float)

    if np.isnan(values).any():
        return np.nan

    x = np.arange(len(values))

    # Linear regression slope
    return np.polyfit(x, values, 1)[0]


df["temperature_trend_6h"] = (
    df.groupby("station_id")["temperature"]
      .rolling(window=6)
      .apply(rolling_slope, raw=True)
      .reset_index(level=0, drop=True)
)

print("6-hour temperature trend created.")

print(
    df[
        ["timestamp", "station_id", "temperature", "temperature_trend_6h"]
    ].head(10)
)

6-hour temperature trend created.
            timestamp  station_id  temperature  temperature_trend_6h
0 2025-01-01 00:00:00       42131          5.4                   NaN
1 2025-01-01 01:00:00       42131          6.1                   NaN
2 2025-01-01 02:00:00       42131          6.1                   NaN
3 2025-01-01 03:00:00       42131          8.0                   NaN
4 2025-01-01 04:00:00       42131          7.6                   NaN
5 2025-01-01 05:00:00       42131          9.8              0.811429
6 2025-01-01 06:00:00       42131         10.0              0.862857
7 2025-01-01 07:00:00       42131         14.9              1.491429
8 2025-01-01 08:00:00       42131         16.3              1.817143
9 2025-01-01 09:00:00       42131         11.8              1.297143


In [52]:
normal_trend = df.loc[
    df["is_anomaly"] == 0,
    "temperature_trend_6h"
].dropna()

print("Normal absolute 6-hour temperature trend:")
print(
    normal_trend.abs().describe(
        percentiles=[0.90, 0.95, 0.99, 0.995, 0.999]
    )
)

Normal absolute 6-hour temperature trend:
count    4.305000e+04
mean     8.124744e-01
std      6.267410e-01
min      5.159021e-19
50%      6.485714e-01
90%      1.717143e+00
95%      2.074286e+00
99%      2.728571e+00
99.5%    2.920000e+00
99.9%    3.325294e+00
max      4.991429e+00
Name: temperature_trend_6h, dtype: float64


In [53]:
# Test possible trend thresholds

for threshold in [1, 1.5, 2, 2.5, 3]:
    count = (normal_trend.abs() > threshold).sum()
    rate = count / len(normal_trend) * 100

    print(
        f"|trend| > {threshold:3}°C/hour : "
        f"{count:5} rows ({rate:.3f}%)"
    )

|trend| >   1°C/hour : 13154 rows (30.555%)
|trend| > 1.5°C/hour :  6299 rows (14.632%)
|trend| >   2°C/hour :  2512 rows (5.835%)
|trend| > 2.5°C/hour :   821 rows (1.907%)
|trend| >   3°C/hour :   154 rows (0.358%)


In [54]:
# Inspect temporal features for all injected anomalies

print(
    df.loc[
        df["is_anomaly"] == 1,
        [
            "timestamp",
            "fault_type",
            "temperature",
            "absolute_temperature_change",
            "repeat_length",
            "temperature_trend_6h"
        ]
    ].to_string(index=False)
)

          timestamp          fault_type  temperature  absolute_temperature_change  repeat_length  temperature_trend_6h
2025-01-05 04:00:00   temperature_spike    55.000000                    44.800000              1          6.380000e+00
2025-01-09 08:00:00 temperature_missing          NaN                          NaN              1                   NaN
2025-01-13 12:00:00  temperature_frozen    16.200000                     0.700000              6          2.142857e-01
2025-01-13 13:00:00  temperature_frozen    16.200000                     0.000000              6         -1.685714e-01
2025-01-13 14:00:00  temperature_frozen    16.200000                     0.000000              6         -3.800000e-01
2025-01-13 15:00:00  temperature_frozen    16.200000                     0.000000              6         -2.314286e-01
2025-01-13 16:00:00  temperature_frozen    16.200000                     0.000000              6         -1.000000e-01
2025-01-13 17:00:00  temperature_frozen    16.20

In [55]:
# Maximum absolute 6-hour trend for each injected fault type

anomaly_temporal = df[df["is_anomaly"] == 1].copy()

print(
    anomaly_temporal.groupby("fault_type")[
        "temperature_trend_6h"
    ].agg(["count", "min", "mean", "max"])
)

                     count       min      mean       max
fault_type                                              
temperature_drift        6 -1.165714 -0.121429  0.931429
temperature_frozen       6 -0.380000 -0.110952  0.214286
temperature_missing      0       NaN       NaN       NaN
temperature_noise        6 -1.039187 -0.786194 -0.433744
temperature_spike        1  6.380000  6.380000  6.380000


In [56]:
# Temporal anomaly detector

df["temporal_score"] = 0

# 1. Sudden temperature change
sudden_change = (
    df["absolute_temperature_change"] > 6
)

# 2. Long repeated temperature sequence
frozen_pattern = (
    df["repeat_length"] >= 6
)

# 3. Extreme 6-hour trend
extreme_trend = (
    df["temperature_trend_6h"].abs() > 3
)

df["temporal_score"] = (
    sudden_change |
    frozen_pattern |
    extreme_trend
).astype(int)

print("Temporal detections:")
print(df["temporal_score"].value_counts())

Temporal detections:
temporal_score
0    42216
1      884
Name: count, dtype: int64


In [57]:
# Temporal detector performance against injected anomalies

temporal_anomalies = df[df["is_anomaly"] == 1]

detected = temporal_anomalies["temporal_score"].sum()
total = len(temporal_anomalies)

print(f"Injected anomalies detected: {detected}/{total}")
print(f"Detection rate: {detected / total * 100:.2f}%")

print("\nDetection by fault type:")
print(
    temporal_anomalies
    .groupby("fault_type")["temporal_score"]
    .agg(["sum", "count"])
)

Injected anomalies detected: 8/20
Detection rate: 40.00%

Detection by fault type:
                     sum  count
fault_type                     
temperature_drift      0      6
temperature_frozen     6      6
temperature_missing    0      1
temperature_noise      1      6
temperature_spike      1      1


In [58]:
# False positives on normal observations

temporal_normal = df[df["is_anomaly"] == 0]

false_positives = temporal_normal["temporal_score"].sum()
normal_count = len(temporal_normal)

print("Normal observations flagged:", false_positives)
print(
    "False-positive rate:",
    f"{false_positives / normal_count * 100:.3f}%"
)

Normal observations flagged: 876
False-positive rate: 2.033%


In [59]:
# Diagnose which temporal rule is causing detections

sudden_change = (
    df["absolute_temperature_change"] > 6
)

frozen_pattern = (
    df["repeat_length"] >= 6
)

extreme_trend = (
    df["temperature_trend_6h"].abs() > 3
)

normal_mask = df["is_anomaly"] == 0

print("Normal false positives by individual rule:")
print()

print(
    "Sudden change > 6°C:",
    sudden_change[normal_mask].sum()
)

print(
    "Repeated value >= 6:",
    frozen_pattern[normal_mask].sum()
)

print(
    "Extreme trend > 3°C/hour:",
    extreme_trend[normal_mask].sum()
)

print("\nCombinations:")
print(
    pd.DataFrame({
        "sudden_change": sudden_change[normal_mask],
        "frozen_pattern": frozen_pattern[normal_mask],
        "extreme_trend": extreme_trend[normal_mask]
    }).value_counts()
)

Normal false positives by individual rule:

Sudden change > 6°C: 176
Repeated value >= 6: 550
Extreme trend > 3°C/hour: 154

Combinations:
sudden_change  frozen_pattern  extreme_trend
False          False           False            42204
               True            False              549
True           False           False              172
False          False           True               151
True           False           True                 3
               True            False                1
Name: count, dtype: int64


In [60]:
# Check how each rule behaves on the 20 injected anomalies

anomaly_mask = df["is_anomaly"] == 1

print(
    pd.DataFrame({
        "fault_type": df.loc[anomaly_mask, "fault_type"],
        "sudden_change": sudden_change[anomaly_mask],
        "frozen_pattern": frozen_pattern[anomaly_mask],
        "extreme_trend": extreme_trend[anomaly_mask]
    }).to_string(index=False)
)

         fault_type  sudden_change  frozen_pattern  extreme_trend
  temperature_spike           True           False           True
temperature_missing          False           False          False
 temperature_frozen          False            True          False
 temperature_frozen          False            True          False
 temperature_frozen          False            True          False
 temperature_frozen          False            True          False
 temperature_frozen          False            True          False
 temperature_frozen          False            True          False
  temperature_drift          False           False          False
  temperature_drift          False           False          False
  temperature_drift          False           False          False
  temperature_drift          False           False          False
  temperature_drift          False           False          False
  temperature_drift          False           False          False
  temperat

In [61]:
# Difference between current temperature and the temperature
# immediately before the repeated run

df["previous_temperature"] = (
    df.groupby("station_id")["temperature"].shift(1)
)

df["repeat_start_difference"] = (
    df["temperature"] - df["previous_temperature"]
).abs()

# Look at normal observations that belong to runs >= 6
normal_long_runs = df[
    (df["is_anomaly"] == 0) &
    (df["repeat_length"] >= 6)
].copy()

print("Normal observations in repeated runs >= 6:")
print(len(normal_long_runs))

print("\nDifference statistics:")
print(
    normal_long_runs["repeat_start_difference"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

Normal observations in repeated runs >= 6:
550

Difference statistics:
count    550.000000
mean       0.178182
std        0.582905
min        0.000000
50%        0.000000
75%        0.000000
90%        1.000000
95%        1.000000
99%        2.000000
max        7.000000
Name: repeat_start_difference, dtype: float64


In [62]:
# Inspect the injected frozen fault

print(
    df[
        (df["is_anomaly"] == 1) &
        (df["fault_type"] == "temperature_frozen")
    ][
        [
            "timestamp",
            "temperature",
            "repeat_length",
            "repeat_start_difference"
        ]
    ].to_string(index=False)
)

          timestamp  temperature  repeat_length  repeat_start_difference
2025-01-13 12:00:00         16.2              6                      0.7
2025-01-13 13:00:00         16.2              6                      0.0
2025-01-13 14:00:00         16.2              6                      0.0
2025-01-13 15:00:00         16.2              6                      0.0
2025-01-13 16:00:00         16.2              6                      0.0
2025-01-13 17:00:00         16.2              6                      0.0


In [63]:
# Calculate rolling temperature variation over 6 hours

df["temperature_std_6h"] = (
    df.groupby("station_id")["temperature"]
      .rolling(window=6)
      .std()
      .reset_index(level=0, drop=True)
)

# Normal observations belonging to long repeated runs
normal_long_runs = df[
    (df["is_anomaly"] == 0) &
    (df["repeat_length"] >= 6)
]

print("Temperature variation during normal long repeated runs:")
print(
    normal_long_runs["temperature_std_6h"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
)

Temperature variation during normal long repeated runs:
count    550.000000
mean       0.669148
std        0.785686
min        0.000000
50%        0.516398
75%        0.894427
90%        1.673320
95%        2.345208
99%        3.624166
max        4.370355
Name: temperature_std_6h, dtype: float64


In [64]:
print(
    df[
        (df["is_anomaly"] == 1) &
        (df["fault_type"] == "temperature_frozen")
    ][
        [
            "timestamp",
            "temperature",
            "repeat_length",
            "temperature_std_6h"
        ]
    ].to_string(index=False)
)

          timestamp  temperature  repeat_length  temperature_std_6h
2025-01-13 12:00:00         16.2              6            1.055304
2025-01-13 13:00:00         16.2              6            0.760044
2025-01-13 14:00:00         16.2              6            0.760044
2025-01-13 15:00:00         16.2              6            0.515429
2025-01-13 16:00:00         16.2              6            0.285774
2025-01-13 17:00:00         16.2              6            0.000000


In [65]:
# Check how often normal long repeated runs become completely flat

normal_long_runs = df[
    (df["is_anomaly"] == 0) &
    (df["repeat_length"] >= 6)
]

for threshold in [0.0, 0.1, 0.2, 0.5]:
    count = (
        normal_long_runs["temperature_std_6h"] <= threshold
    ).sum()

    rate = count / len(df[df["is_anomaly"] == 0]) * 100

    print(
        f"6-hour std <= {threshold}: "
        f"{count} rows ({rate:.3f}% of normal data)"
    )

6-hour std <= 0.0: 190 rows (0.441% of normal data)
6-hour std <= 0.1: 190 rows (0.441% of normal data)
6-hour std <= 0.2: 190 rows (0.441% of normal data)
6-hour std <= 0.5: 261 rows (0.606% of normal data)


In [66]:
# Check the final frozen observation

frozen_rows = df[
    (df["is_anomaly"] == 1) &
    (df["fault_type"] == "temperature_frozen")
]

print(
    frozen_rows[
        ["timestamp", "repeat_length", "temperature_std_6h"]
    ].to_string(index=False)
)

          timestamp  repeat_length  temperature_std_6h
2025-01-13 12:00:00              6            1.055304
2025-01-13 13:00:00              6            0.760044
2025-01-13 14:00:00              6            0.760044
2025-01-13 15:00:00              6            0.515429
2025-01-13 16:00:00              6            0.285774
2025-01-13 17:00:00              6            0.000000


In [67]:
# Improved temporal anomaly detector

# 1. Sudden temperature change
sudden_change = (
    df["absolute_temperature_change"] > 6
)

# 2. Frozen sensor pattern:
#    long repeated sequence AND nearly zero variation
frozen_pattern = (
    (df["repeat_length"] >= 6) &
    (df["temperature_std_6h"] <= 0.1)
)

# 3. Extreme 6-hour trend
extreme_trend = (
    df["temperature_trend_6h"].abs() > 3
)

# Combine temporal signals
df["temporal_score"] = (
    sudden_change |
    frozen_pattern |
    extreme_trend
).astype(int)

print("Temporal detections:")
print(df["temporal_score"].value_counts())

Temporal detections:
temporal_score
0    42580
1      520
Name: count, dtype: int64


In [68]:
# Temporal detector performance

temporal_anomalies = df[df["is_anomaly"] == 1]

detected = temporal_anomalies["temporal_score"].sum()
total = len(temporal_anomalies)

print(f"Injected anomalies detected: {detected}/{total}")
print(f"Detection rate: {detected / total * 100:.2f}%")

print("\nDetection by fault type:")

print(
    temporal_anomalies
    .groupby("fault_type")["temporal_score"]
    .agg(["sum", "count"])
)

Injected anomalies detected: 3/20
Detection rate: 15.00%

Detection by fault type:
                     sum  count
fault_type                     
temperature_drift      0      6
temperature_frozen     1      6
temperature_missing    0      1
temperature_noise      1      6
temperature_spike      1      1


In [69]:
# False positives on normal observations

temporal_normal = df[df["is_anomaly"] == 0]

false_positives = temporal_normal["temporal_score"].sum()
normal_count = len(temporal_normal)

print("Normal observations flagged:", false_positives)

print(
    "False-positive rate:",
    f"{false_positives / normal_count * 100:.3f}%"
)

Normal observations flagged: 517
False-positive rate: 1.200%


In [70]:
# Inspect all detector scores for injected anomalies

anomaly_check = df[df["is_anomaly"] == 1].copy()

print(
    anomaly_check[
        [
            "timestamp",
            "fault_type",
            "rule_score",
            "statistical_score",
            "isolation_score",
            "temporal_score"
        ]
    ].to_string(index=False)
)

          timestamp          fault_type  rule_score  statistical_score  isolation_score  temporal_score
2025-01-05 04:00:00   temperature_spike           0                  0                0               1
2025-01-09 08:00:00 temperature_missing           1                  0                0               0
2025-01-13 12:00:00  temperature_frozen           0                  0                1               0
2025-01-13 13:00:00  temperature_frozen           0                  0                0               0
2025-01-13 14:00:00  temperature_frozen           0                  0                0               0
2025-01-13 15:00:00  temperature_frozen           0                  0                0               0
2025-01-13 16:00:00  temperature_frozen           0                  0                0               0
2025-01-13 17:00:00  temperature_frozen           0                  0                0               1
2025-01-17 16:00:00   temperature_drift           0             

In [71]:
# Count how many detectors flag each injected anomaly

anomaly_check["detector_count"] = (
    anomaly_check["rule_score"]
    + anomaly_check["statistical_score"]
    + anomaly_check["isolation_score"]
    + anomaly_check["temporal_score"]
)

print("\nDetector-count distribution among injected anomalies:")
print(
    anomaly_check["detector_count"]
    .value_counts()
    .sort_index()
)

print("\nAverage detector count by fault type:")
print(
    anomaly_check
    .groupby("fault_type")["detector_count"]
    .agg(["count", "mean", "min", "max"])
)


Detector-count distribution among injected anomalies:
detector_count
0    15
1     4
2     1
Name: count, dtype: int64

Average detector count by fault type:
                     count      mean  min  max
fault_type                                    
temperature_drift        6  0.000000    0    0
temperature_frozen       6  0.333333    0    1
temperature_missing      1  1.000000    1    1
temperature_noise        6  0.333333    0    2
temperature_spike        1  1.000000    1    1


In [72]:
# Create combined detector score

df["anomaly_score"] = (
    df["rule_score"]
    + df["statistical_score"]
    + df["isolation_score"]
    + df["temporal_score"]
)

print("Anomaly score distribution:")
print(
    df["anomaly_score"]
    .value_counts()
    .sort_index()
)

Anomaly score distribution:
anomaly_score
0    35536
1     7257
2      297
3       10
Name: count, dtype: int64


In [73]:
# Evaluate different anomaly-score thresholds

for threshold in [1, 2, 3]:

    df["test_prediction"] = (
        df["anomaly_score"] >= threshold
    ).astype(int)

    y_true = df["is_anomaly"]
    y_pred = df["test_prediction"]

    print(f"\n===== Threshold >= {threshold} =====")

    print("Predicted anomalies:", y_pred.sum())

    print(
        "True injected anomalies detected:",
        ((y_pred == 1) & (y_true == 1)).sum(),
        "/",
        y_true.sum()
    )

    print(
        "Recall:",
        f"{recall_score(y_true, y_pred):.4f}"
    )

    print(
        "Precision:",
        f"{precision_score(y_true, y_pred, zero_division=0):.4f}"
    )

    print(
        "F1:",
        f"{f1_score(y_true, y_pred, zero_division=0):.4f}"
    )


===== Threshold >= 1 =====
Predicted anomalies: 7564
True injected anomalies detected: 5 / 20
Recall: 0.2500
Precision: 0.0007
F1: 0.0013

===== Threshold >= 2 =====
Predicted anomalies: 307
True injected anomalies detected: 1 / 20
Recall: 0.0500
Precision: 0.0033
F1: 0.0061

===== Threshold >= 3 =====
Predicted anomalies: 10
True injected anomalies detected: 0 / 20
Recall: 0.0000
Precision: 0.0000
F1: 0.0000


In [74]:
# Inspect temperature values around each injected anomaly

station_data = df[df["station_id"] == 42182].copy()
station_data = station_data.sort_values("timestamp")

anomaly_times = station_data.loc[
    station_data["is_anomaly"] == 1,
    "timestamp"
].tolist()

for anomaly_time in anomaly_times:

    window = station_data[
        (station_data["timestamp"] >= anomaly_time - pd.Timedelta(hours=3)) &
        (station_data["timestamp"] <= anomaly_time + pd.Timedelta(hours=3))
    ][
        [
            "timestamp",
            "temperature",
            "temperature_rolling_mean_24h",
            "temperature_deviation_24h",
            "temperature_zscore_24h",
            "is_anomaly",
            "fault_type"
        ]
    ]

    print("\n" + "=" * 70)
    print("Anomaly:", anomaly_time)
    print(window.to_string(index=False))


Anomaly: 2025-01-05 04:00:00
          timestamp  temperature  temperature_rolling_mean_24h  temperature_deviation_24h  temperature_zscore_24h  is_anomaly        fault_type
2025-01-05 01:00:00         10.7                     13.254167                  -2.554167               -0.762140           0            normal
2025-01-05 02:00:00         10.9                     13.333333                  -2.433333               -0.745783           0            normal
2025-01-05 03:00:00         10.2                     13.408333                  -3.208333               -1.008621           0            normal
2025-01-05 04:00:00         55.0                     13.375000                  -1.975000               -0.615017           1 temperature_spike
2025-01-05 05:00:00         12.7                     13.333333                  -0.633333               -0.196023           0            normal
2025-01-05 06:00:00         13.0                     13.291667                  -0.291667               -0

In [75]:
# Create a past-only 6-hour rolling median baseline

previous_temperature = (
    df.groupby("station_id")["temperature"]
      .shift(1)
)

df["previous_6h_median"] = (
    previous_temperature
    .groupby(df["station_id"])
    .rolling(window=6, min_periods=3)
    .median()
    .reset_index(level=0, drop=True)
)

df["local_temperature_residual"] = (
    df["temperature"] - df["previous_6h_median"]
).abs()

print("Past-only baseline created.")

print("\nSample:")
print(
    df[
        [
            "timestamp",
            "station_id",
            "temperature",
            "previous_6h_median",
            "local_temperature_residual"
        ]
    ].head(15)
)

Past-only baseline created.

Sample:
             timestamp  station_id  temperature  previous_6h_median  \
0  2025-01-01 00:00:00       42131          5.4                 NaN   
1  2025-01-01 01:00:00       42131          6.1                 NaN   
2  2025-01-01 02:00:00       42131          6.1                 NaN   
3  2025-01-01 03:00:00       42131          8.0                6.10   
4  2025-01-01 04:00:00       42131          7.6                6.10   
5  2025-01-01 05:00:00       42131          9.8                6.10   
6  2025-01-01 06:00:00       42131         10.0                6.85   
7  2025-01-01 07:00:00       42131         14.9                7.80   
8  2025-01-01 08:00:00       42131         16.3                8.90   
9  2025-01-01 09:00:00       42131         11.8                9.90   
10 2025-01-01 10:00:00       42131         17.1               10.90   
11 2025-01-01 11:00:00       42131         16.4               13.35   
12 2025-01-01 12:00:00       42131      

In [76]:
# Compare local residuals for normal and injected anomaly observations

normal_residuals = df.loc[
    df["is_anomaly"] == 0,
    "local_temperature_residual"
].dropna()

anomaly_residuals = df.loc[
    df["is_anomaly"] == 1,
    "local_temperature_residual"
].dropna()

print("Normal observations:")
print(normal_residuals.describe())

print("\nInjected anomalies:")
print(anomaly_residuals.describe())

print("\nInjected anomalies by fault type:")
print(
    df[df["is_anomaly"] == 1]
    .groupby("fault_type")["local_temperature_residual"]
    .agg(["count", "mean", "min", "median", "max"])
)

Normal observations:
count    43065.000000
mean         2.839607
std          2.297826
min          0.000000
25%          1.100000
50%          2.200000
75%          4.000000
max         16.650000
Name: local_temperature_residual, dtype: float64

Injected anomalies:
count    19.000000
mean      4.188734
std       9.982795
min       0.000000
25%       0.350000
50%       1.190566
75%       4.050000
max      44.550000
Name: local_temperature_residual, dtype: float64

Injected anomalies by fault type:
                     count       mean        min     median        max
fault_type                                                            
temperature_drift        6   2.208333   0.400000   1.875000   4.050000
temperature_frozen       6   0.233333   0.000000   0.350000   0.350000
temperature_missing      0        NaN        NaN        NaN        NaN
temperature_noise        6   3.397658   1.023588   2.874595   6.742635
temperature_spike        1  44.550000  44.550000  44.550000  44.550000


In [77]:
# Compare short-term temperature variability
# between normal observations and injected anomalies

normal_std = df.loc[
    df["is_anomaly"] == 0,
    "temperature_std_6h"
].dropna()

anomaly_std = df.loc[
    df["is_anomaly"] == 1,
    "temperature_std_6h"
].dropna()

print("Normal observations:")
print(normal_std.describe())

print("\nInjected anomalies:")
print(anomaly_std.describe())

print("\nInjected anomalies by fault type:")
print(
    df[df["is_anomaly"] == 1]
    .groupby("fault_type")["temperature_std_6h"]
    .agg(["count", "mean", "min", "median", "max"])
)

Normal observations:
count    43050.000000
mean         1.721201
std          1.193447
min          0.000000
25%          0.836660
50%          1.382329
75%          2.338090
max         18.029171
Name: temperature_std_6h, dtype: float64

Injected anomalies:
count    19.000000
mean      2.372211
std       3.925118
min       0.000000
25%       0.842600
50%       1.597081
75%       2.220059
max      18.186304
Name: temperature_std_6h, dtype: float64

Injected anomalies by fault type:
                     count       mean        min     median        max
fault_type                                                            
temperature_drift        6   1.667117   1.195687   1.723022   2.240238
temperature_frozen       6   0.562766   0.000000   0.637736   1.055304
temperature_missing      0        NaN        NaN        NaN        NaN
temperature_noise        6   2.251068   0.925156   2.295114   3.022704
temperature_spike        1  18.186304  18.186304  18.186304  18.186304


In [78]:
# Measure directional persistence over the previous 6 hours

df["temperature_change"] = (
    df.groupby("station_id")["temperature"].diff()
)

df["change_direction"] = np.sign(df["temperature_change"])

df["positive_changes_6h"] = (
    df.groupby("station_id")["change_direction"]
      .rolling(window=6, min_periods=3)
      .apply(lambda x: np.sum(x > 0), raw=True)
      .reset_index(level=0, drop=True)
)

df["negative_changes_6h"] = (
    df.groupby("station_id")["change_direction"]
      .rolling(window=6, min_periods=3)
      .apply(lambda x: np.sum(x < 0), raw=True)
      .reset_index(level=0, drop=True)
)

print("Directional features created.")

print(
    df[
        [
            "timestamp",
            "temperature",
            "temperature_change",
            "positive_changes_6h",
            "negative_changes_6h"
        ]
    ].head(15)
)

Directional features created.
             timestamp  temperature  temperature_change  positive_changes_6h  \
0  2025-01-01 00:00:00          5.4                 NaN                  NaN   
1  2025-01-01 01:00:00          6.1                 0.7                  NaN   
2  2025-01-01 02:00:00          6.1                 0.0                  NaN   
3  2025-01-01 03:00:00          8.0                 1.9                  2.0   
4  2025-01-01 04:00:00          7.6                -0.4                  2.0   
5  2025-01-01 05:00:00          9.8                 2.2                  3.0   
6  2025-01-01 06:00:00         10.0                 0.2                  4.0   
7  2025-01-01 07:00:00         14.9                 4.9                  4.0   
8  2025-01-01 08:00:00         16.3                 1.4                  5.0   
9  2025-01-01 09:00:00         11.8                -4.5                  4.0   
10 2025-01-01 10:00:00         17.1                 5.3                  5.0   
11 2025-01

In [79]:
# Compare directional persistence for normal observations
# and injected drift anomalies

normal_positive = df.loc[
    df["is_anomaly"] == 0,
    "positive_changes_6h"
].dropna()

normal_negative = df.loc[
    df["is_anomaly"] == 0,
    "negative_changes_6h"
].dropna()

drift_data = df[
    df["fault_type"] == "temperature_drift"
]

print("Normal positive changes:")
print(normal_positive.describe())

print("\nNormal negative changes:")
print(normal_negative.describe())

print("\nInjected drift observations:")
print(
    drift_data[
        [
            "timestamp",
            "temperature",
            "temperature_change",
            "positive_changes_6h",
            "negative_changes_6h"
        ]
    ].to_string(index=False)
)

Normal positive changes:
count    43065.000000
mean         2.195542
std          2.077756
min          0.000000
25%          0.000000
50%          2.000000
75%          4.000000
max          6.000000
Name: positive_changes_6h, dtype: float64

Normal negative changes:
count    43065.000000
mean         3.235295
std          2.129388
min          0.000000
25%          1.000000
50%          3.000000
75%          5.000000
max          6.000000
Name: negative_changes_6h, dtype: float64

Injected drift observations:
          timestamp  temperature  temperature_change  positive_changes_6h  negative_changes_6h
2025-01-17 16:00:00         11.1                -0.3                  0.0                  6.0
2025-01-17 17:00:00         11.9                 0.8                  1.0                  5.0
2025-01-17 18:00:00         13.8                 1.9                  2.0                  4.0
2025-01-17 19:00:00         13.2                -0.6                  2.0                  4.0
2025-01-

In [80]:
# Create a past-only 12-hour rolling median baseline

previous_temperature = (
    df.groupby("station_id")["temperature"].shift(1)
)

df["previous_12h_median"] = (
    previous_temperature
    .groupby(df["station_id"])
    .rolling(window=12, min_periods=6)
    .median()
    .reset_index(level=0, drop=True)
)

df["local_residual_12h"] = (
    df["temperature"] - df["previous_12h_median"]
).abs()

print("Past-only 12-hour baseline created.")

print(
    df[
        [
            "timestamp",
            "temperature",
            "previous_12h_median",
            "local_residual_12h"
        ]
    ].head(20)
)

Past-only 12-hour baseline created.
             timestamp  temperature  previous_12h_median  local_residual_12h
0  2025-01-01 00:00:00          5.4                  NaN                 NaN
1  2025-01-01 01:00:00          6.1                  NaN                 NaN
2  2025-01-01 02:00:00          6.1                  NaN                 NaN
3  2025-01-01 03:00:00          8.0                  NaN                 NaN
4  2025-01-01 04:00:00          7.6                  NaN                 NaN
5  2025-01-01 05:00:00          9.8                  NaN                 NaN
6  2025-01-01 06:00:00         10.0                 6.85                3.15
7  2025-01-01 07:00:00         14.9                 7.60                7.30
8  2025-01-01 08:00:00         16.3                 7.80                8.50
9  2025-01-01 09:00:00         11.8                 8.00                3.80
10 2025-01-01 10:00:00         17.1                 8.90                8.20
11 2025-01-01 11:00:00         16.4     

In [81]:
# Compare 12-hour contextual residuals
# for normal observations and injected anomalies

normal_12h = df.loc[
    df["is_anomaly"] == 0,
    "local_residual_12h"
].dropna()

anomaly_12h = df.loc[
    df["is_anomaly"] == 1,
    "local_residual_12h"
].dropna()

print("Normal observations:")
print(normal_12h.describe())

print("\nInjected anomalies:")
print(anomaly_12h.describe())

print("\nInjected anomalies by fault type:")
print(
    df[df["is_anomaly"] == 1]
    .groupby("fault_type")["local_residual_12h"]
    .agg(["count", "mean", "min", "median", "max"])
)

Normal observations:
count    43050.000000
mean         4.279248
std          2.977499
min          0.000000
25%          2.000000
50%          3.700000
75%          5.900000
max         16.700000
Name: local_residual_12h, dtype: float64

Injected anomalies:
count    19.000000
mean      5.005245
std       9.811591
min       0.000000
25%       1.000000
50%       2.800000
75%       4.245283
max      44.300000
Name: local_residual_12h, dtype: float64

Injected anomalies by fault type:
                     count       mean        min     median        max
fault_type                                                            
temperature_drift        6   2.341667   0.650000   2.375000   4.050000
temperature_frozen       6   0.866667   0.000000   0.300000   2.900000
temperature_missing      0        NaN        NaN        NaN        NaN
temperature_noise        6   5.258275   2.318871   4.999821   8.029968
temperature_spike        1  44.300000  44.300000  44.300000  44.300000


In [82]:
# Compare recent temperature trend with the preceding trend

def slope(values):
    values = np.array(values, dtype=float)

    if np.isnan(values).any():
        return np.nan

    x = np.arange(len(values))
    return np.polyfit(x, values, 1)[0]


# Trend over the most recent 3 hours
df["trend_recent_3h"] = (
    df.groupby("station_id")["temperature"]
      .rolling(window=3, min_periods=3)
      .apply(slope, raw=True)
      .reset_index(level=0, drop=True)
)

# Trend over the 3 hours before that
df["trend_previous_3h"] = (
    df.groupby("station_id")["temperature"]
      .shift(3)
      .groupby(df["station_id"])
      .rolling(window=3, min_periods=3)
      .apply(slope, raw=True)
      .reset_index(level=0, drop=True)
)

# Change in trajectory
df["trend_change_3h"] = (
    df["trend_recent_3h"] - df["trend_previous_3h"]
).abs()

print("Trajectory features created.")

print(
    df[
        [
            "timestamp",
            "temperature",
            "trend_recent_3h",
            "trend_previous_3h",
            "trend_change_3h"
        ]
    ].head(20)
)

Trajectory features created.
             timestamp  temperature  trend_recent_3h  trend_previous_3h  \
0  2025-01-01 00:00:00          5.4              NaN                NaN   
1  2025-01-01 01:00:00          6.1              NaN                NaN   
2  2025-01-01 02:00:00          6.1             0.35                NaN   
3  2025-01-01 03:00:00          8.0             0.95                NaN   
4  2025-01-01 04:00:00          7.6             0.75                NaN   
5  2025-01-01 05:00:00          9.8             0.90               0.35   
6  2025-01-01 06:00:00         10.0             1.20               0.95   
7  2025-01-01 07:00:00         14.9             2.55               0.75   
8  2025-01-01 08:00:00         16.3             3.15               0.90   
9  2025-01-01 09:00:00         11.8            -1.55               1.20   
10 2025-01-01 10:00:00         17.1             0.40               2.55   
11 2025-01-01 11:00:00         16.4             2.30               3.15

In [83]:
# Compare trajectory-change values for normal data and drift anomalies

normal_traj = df.loc[
    df["is_anomaly"] == 0,
    "trend_change_3h"
].dropna()

drift_traj = df.loc[
    df["fault_type"] == "temperature_drift",
    "trend_change_3h"
].dropna()

print("Normal observations:")
print(normal_traj.describe())

print("\nInjected drift observations:")
print(drift_traj.describe())

print("\nDrift observations:")
print(
    df[df["fault_type"] == "temperature_drift"][
        [
            "timestamp",
            "temperature",
            "trend_recent_3h",
            "trend_previous_3h",
            "trend_change_3h"
        ]
    ].to_string(index=False)
)

Normal observations:
count    43050.000000
mean         1.019171
std          1.024777
min          0.000000
25%          0.300000
50%          0.750000
75%          1.500000
max         21.950000
Name: trend_change_3h, dtype: float64

Injected drift observations:
count    6.000000
mean     1.208333
std      1.159059
min      0.250000
25%      0.387500
50%      0.750000
75%      1.712500
max      3.200000
Name: trend_change_3h, dtype: float64

Drift observations:
          timestamp  temperature  trend_recent_3h  trend_previous_3h  trend_change_3h
2025-01-17 16:00:00         11.1    -1.300000e+00              -0.80             0.50
2025-01-17 17:00:00         11.9     2.500000e-01              -0.75             1.00
2025-01-17 18:00:00         13.8     1.350000e+00              -1.85             3.20
2025-01-17 19:00:00         13.2     6.500000e-01              -1.30             1.95
2025-01-17 20:00:00         13.8    -1.843635e-16               0.25             0.25
2025-01-17 21:00

In [84]:
# Test local variability for injected noise anomalies

normal_std = df.loc[
    df["is_anomaly"] == 0,
    "temperature_std_6h"
].dropna()

noise_std = df.loc[
    df["fault_type"] == "temperature_noise",
    "temperature_std_6h"
].dropna()

print("Normal observations:")
print(normal_std.describe())

print("\nInjected noise observations:")
print(noise_std.describe())

print("\nNoise observations:")
print(
    df[df["fault_type"] == "temperature_noise"][
        [
            "timestamp",
            "temperature",
            "temperature_change",
            "temperature_std_6h",
            "local_temperature_residual",
            "local_residual_12h"
        ]
    ].to_string(index=False)
)

Normal observations:
count    43050.000000
mean         1.721201
std          1.193447
min          0.000000
25%          0.836660
50%          1.382329
75%          2.338090
max         18.029171
Name: temperature_std_6h, dtype: float64

Injected noise observations:
count    6.000000
mean     2.251068
std      0.768376
min      0.925156
25%      2.050342
50%      2.295114
75%      2.823455
max      3.022704
Name: temperature_std_6h, dtype: float64

Noise observations:
          timestamp  temperature  temperature_change  temperature_std_6h  local_temperature_residual  local_residual_12h
2025-01-21 20:00:00    13.909434            0.209434            0.925156                    1.190566            4.440566
2025-01-21 21:00:00     8.720032           -5.189402            2.390349                    5.679968            8.029968
2025-01-21 22:00:00    12.400902            3.680871            2.199880                    1.553815            3.299098
2025-01-21 23:00:00    12.781129          

In [85]:
# Test combined local variability + sudden-change signal for noise

noise_test = df[df["fault_type"] == "temperature_noise"].copy()
normal_test = df[df["is_anomaly"] == 0].copy()

conditions = {
    "std > 2": df["temperature_std_6h"] > 2,
    "std > 2.5": df["temperature_std_6h"] > 2.5,
    "change > 4": df["absolute_temperature_change"] > 4,
    "change > 5": df["absolute_temperature_change"] > 5,
    "residual > 4": df["local_temperature_residual"] > 4,
    "residual > 5": df["local_temperature_residual"] > 5,
}

print("Detection rates on injected noise:")
for name, condition in conditions.items():
    detected = condition[noise_test.index].sum()
    print(f"{name}: {detected}/6")

print("\nCombined conditions:")
for name, condition in {
    "std > 2 AND change > 4":
        (df["temperature_std_6h"] > 2) &
        (df["absolute_temperature_change"] > 4),

    "std > 2 AND residual > 4":
        (df["temperature_std_6h"] > 2) &
        (df["local_temperature_residual"] > 4),

    "change > 4 AND residual > 4":
        (df["absolute_temperature_change"] > 4) &
        (df["local_temperature_residual"] > 4),

    "std > 2 AND change > 4 AND residual > 4":
        (df["temperature_std_6h"] > 2) &
        (df["absolute_temperature_change"] > 4) &
        (df["local_temperature_residual"] > 4)
}.items():

    noise_detected = condition[noise_test.index].sum()
    normal_fp = condition[normal_test.index].sum()

    print(
        f"{name}: "
        f"noise={noise_detected}/6, "
        f"normal_FPs={normal_fp}"
    )

Detection rates on injected noise:
std > 2: 5/6
std > 2.5: 2/6
change > 4: 2/6
change > 5: 2/6
residual > 4: 3/6
residual > 5: 2/6

Combined conditions:
std > 2 AND change > 4: noise=2/6, normal_FPs=588
std > 2 AND residual > 4: noise=3/6, normal_FPs=9988
change > 4 AND residual > 4: noise=2/6, normal_FPs=564
std > 2 AND change > 4 AND residual > 4: noise=2/6, normal_FPs=524


In [86]:
# Test whether frozen periods have unusually low local variability
# compared with normal observations

normal_std6 = df.loc[
    df["is_anomaly"] == 0,
    "temperature_std_6h"
].dropna()

frozen_std6 = df.loc[
    df["fault_type"] == "temperature_frozen",
    "temperature_std_6h"
].dropna()

print("Normal observations:")
print(normal_std6.describe())

print("\nInjected frozen observations:")
print(frozen_std6.describe())

print("\nFrozen observations:")
print(
    df[df["fault_type"] == "temperature_frozen"][
        [
            "timestamp",
            "temperature",
            "repeat_length",
            "temperature_std_6h",
            "temperature_change",
            "local_temperature_residual"
        ]
    ].to_string(index=False)
)

Normal observations:
count    43050.000000
mean         1.721201
std          1.193447
min          0.000000
25%          0.836660
50%          1.382329
75%          2.338090
max         18.029171
Name: temperature_std_6h, dtype: float64

Injected frozen observations:
count    6.000000
mean     0.562766
std      0.378445
min      0.000000
25%      0.343188
50%      0.637736
75%      0.760044
max      1.055304
Name: temperature_std_6h, dtype: float64

Frozen observations:
          timestamp  temperature  repeat_length  temperature_std_6h  temperature_change  local_temperature_residual
2025-01-13 12:00:00         16.2              6            1.055304                -0.7                        0.35
2025-01-13 13:00:00         16.2              6            0.760044                 0.0                        0.35
2025-01-13 14:00:00         16.2              6            0.760044                 0.0                        0.35
2025-01-13 15:00:00         16.2              6            0

In [87]:
# Test different frozen-pattern thresholds

normal_test = df[df["is_anomaly"] == 0]
frozen_test = df[df["fault_type"] == "temperature_frozen"]

tests = {
    "repeat >= 6 AND std <= 1.0":
        (df["repeat_length"] >= 6) &
        (df["temperature_std_6h"] <= 1.0),

    "repeat >= 6 AND std <= 0.75":
        (df["repeat_length"] >= 6) &
        (df["temperature_std_6h"] <= 0.75),

    "repeat >= 6 AND std <= 0.5":
        (df["repeat_length"] >= 6) &
        (df["temperature_std_6h"] <= 0.5),

    "repeat >= 6 AND std <= 0.25":
        (df["repeat_length"] >= 6) &
        (df["temperature_std_6h"] <= 0.25),
}

for name, condition in tests.items():

    frozen_detected = condition[frozen_test.index].sum()
    normal_fp = condition[normal_test.index].sum()

    print(
        f"{name}\n"
        f"  Frozen detected: {frozen_detected}/6\n"
        f"  Normal false positives: {normal_fp}\n"
    )

repeat >= 6 AND std <= 1.0
  Frozen detected: 5/6
  Normal false positives: 437

repeat >= 6 AND std <= 0.75
  Frozen detected: 3/6
  Normal false positives: 346

repeat >= 6 AND std <= 0.5
  Frozen detected: 2/6
  Normal false positives: 261

repeat >= 6 AND std <= 0.25
  Frozen detected: 1/6
  Normal false positives: 190



In [88]:
# Update the temporal detector with the improved frozen-pattern rule

sudden_change = (
    df["absolute_temperature_change"] > 6
)

frozen_pattern = (
    (df["repeat_length"] >= 6) &
    (df["temperature_std_6h"] <= 1.0)
)

extreme_trend = (
    df["temperature_trend_6h"].abs() > 3
)

df["temporal_score"] = (
    sudden_change |
    frozen_pattern |
    extreme_trend
).astype(int)

print("Temporal detections:", df["temporal_score"].sum())

print("\nInjected anomalies detected:")
print(
    df.loc[df["is_anomaly"] == 1, "temporal_score"].sum(),
    "/",
    df["is_anomaly"].sum()
)

print("\nDetection by fault type:")
print(
    df[df["is_anomaly"] == 1]
    .groupby("fault_type")["temporal_score"]
    .agg(["sum", "count"])
)

Temporal detections: 771

Injected anomalies detected:
7 / 20

Detection by fault type:
                     sum  count
fault_type                     
temperature_drift      0      6
temperature_frozen     5      6
temperature_missing    0      1
temperature_noise      1      6
temperature_spike      1      1


In [89]:
# Recalculate the combined anomaly score
# using the updated temporal detector

df["anomaly_score"] = (
    df["rule_score"]
    + df["statistical_score"]
    + df["isolation_score"]
    + df["temporal_score"]
)

print("Anomaly score distribution:")
print(df["anomaly_score"].value_counts().sort_index())

print("\nScores for injected anomalies:")
print(
    df[df["is_anomaly"] == 1][
        [
            "timestamp",
            "fault_type",
            "rule_score",
            "statistical_score",
            "isolation_score",
            "temporal_score",
            "anomaly_score"
        ]
    ].to_string(index=False)
)

Anomaly score distribution:
anomaly_score
0    35308
1     7462
2      320
3       10
Name: count, dtype: int64

Scores for injected anomalies:
          timestamp          fault_type  rule_score  statistical_score  isolation_score  temporal_score  anomaly_score
2025-01-05 04:00:00   temperature_spike           0                  0                0               1              1
2025-01-09 08:00:00 temperature_missing           1                  0                0               0              1
2025-01-13 12:00:00  temperature_frozen           0                  0                1               0              1
2025-01-13 13:00:00  temperature_frozen           0                  0                0               1              1
2025-01-13 14:00:00  temperature_frozen           0                  0                0               1              1
2025-01-13 15:00:00  temperature_frozen           0                  0                0               1              1
2025-01-13 16:00:00  te

In [90]:
# Evaluate each detector independently

y_true = df["is_anomaly"]

detectors = {
    "Rule-Based": df["rule_score"],
    "Statistical": df["statistical_score"],
    "Isolation Forest": df["isolation_score"],
    "Temporal": df["temporal_score"]
}

for name, prediction in detectors.items():

    print(f"\n===== {name} =====")

    print("Predicted anomalies:", prediction.sum())

    print(
        "Injected anomalies detected:",
        ((prediction == 1) & (y_true == 1)).sum(),
        "/",
        y_true.sum()
    )

    print(
        "Recall:",
        f"{recall_score(y_true, prediction, zero_division=0):.4f}"
    )

    print(
        "Precision:",
        f"{precision_score(y_true, prediction, zero_division=0):.4f}"
    )

    print(
        "F1:",
        f"{f1_score(y_true, prediction, zero_division=0):.4f}"
    )


===== Rule-Based =====
Predicted anomalies: 1
Injected anomalies detected: 1 / 20
Recall: 0.0500
Precision: 1.0000
F1: 0.0952

===== Statistical =====
Predicted anomalies: 148
Injected anomalies detected: 0 / 20
Recall: 0.0000
Precision: 0.0000
F1: 0.0000

===== Isolation Forest =====
Predicted anomalies: 7212
Injected anomalies detected: 2 / 20
Recall: 0.1000
Precision: 0.0003
F1: 0.0006

===== Temporal =====
Predicted anomalies: 771
Injected anomalies detected: 7 / 20
Recall: 0.3500
Precision: 0.0091
F1: 0.0177


In [91]:
# Prepare final Person 2 results

result_columns = [
    "timestamp",
    "station_id",
    "station_name",

    "temperature",
    "humidity",
    "pressure",
    "wind_speed",

    "temperature_deviation_24h",
    "temperature_zscore_24h",

    "temperature_change",
    "absolute_temperature_change",
    "temperature_std_6h",
    "temperature_trend_6h",

    "previous_6h_median",
    "local_temperature_residual",
    "previous_12h_median",
    "local_residual_12h",

    "repeat_length",

    "rule_score",
    "statistical_score",
    "isolation_score",
    "temporal_score",
    "anomaly_score",

    # Ground truth — evaluation only
    "fault_type",
    "fault_severity",
    "is_anomaly"
]

results_df = df[result_columns].copy()

print("Final results shape:", results_df.shape)
print("\nColumns:")
print(results_df.columns.tolist())

print("\nAnomaly rows:")
print(
    results_df[results_df["is_anomaly"] == 1][
        [
            "timestamp",
            "fault_type",
            "rule_score",
            "statistical_score",
            "isolation_score",
            "temporal_score",
            "anomaly_score"
        ]
    ].to_string(index=False)
)

Final results shape: (43100, 26)

Columns:
['timestamp', 'station_id', 'station_name', 'temperature', 'humidity', 'pressure', 'wind_speed', 'temperature_deviation_24h', 'temperature_zscore_24h', 'temperature_change', 'absolute_temperature_change', 'temperature_std_6h', 'temperature_trend_6h', 'previous_6h_median', 'local_temperature_residual', 'previous_12h_median', 'local_residual_12h', 'repeat_length', 'rule_score', 'statistical_score', 'isolation_score', 'temporal_score', 'anomaly_score', 'fault_type', 'fault_severity', 'is_anomaly']

Anomaly rows:
          timestamp          fault_type  rule_score  statistical_score  isolation_score  temporal_score  anomaly_score
2025-01-05 04:00:00   temperature_spike           0                  0                0               1              1
2025-01-09 08:00:00 temperature_missing           1                  0                0               0              1
2025-01-13 12:00:00  temperature_frozen           0                  0               

In [94]:
# Save Person 2 results to Google Drive

results_df.to_csv(RESULTS_PATH, index=False)

print("Person 2 results saved successfully!")
print("Path:", RESULTS_PATH)

# Verify the saved file
saved_df = pd.read_csv(RESULTS_PATH)

print("\nSaved file shape:", saved_df.shape)
print("Saved file exists:", os.path.exists(RESULTS_PATH))

Person 2 results saved successfully!
Path: /content/drive/MyDrive/SATARK_AI/outputs/results/person2_day2_results.csv

Saved file shape: (43100, 26)
Saved file exists: True
